# **A primer on inference and analysis of microbial association networks from metataxonomic data.**

Prof. Eugenio Parente, Università degli Studi della Basilicata

Sometime in June 2026

# What is this notebook for?  


This notebook is a proof of concept for the installation and use of R packages needed for the inference and analysis of Microbial Association Networks, focused on the use of the [NetCoMi](https://netcomi.de/) package, sesigned for my lecture at this [Elixir training course](https://elixir-iib-training.github.io/site/2026-06-29-Profiling_microbial_communities).  
Microbial Association Networks (MAN) have been used for more than 10 years with two main purposes:  

1.   providing information about potential interactions of taxa in microbiomes
2.   exploring emerging properties of microbiomes

Background on inferemnce of MAN and MAN inference can be found in the following papers:  

1.   Kajihara et al. (2024): excellent review, gives a good prerspective on methods and potential uses of network analysis
2.   Kurtz et al. (2015): describes the SPIEC-EASI algorithm
2.   Liu et al. (2023): an excellent review on network comparison
3.   Parente et al., (2022): reviews methods for inference and analysis of microbial association networks with emphasis on cheese
3.   Peschel et al., (2020): introduces the NetCoMi package and reviews concepts and methods on network inference

The "Orchestrating Microbiome Analaysis" site offers a thorough presentation of [this matter](https://microbiome.github.io/OMA/docs/devel/pages/network_learning.html).
The notebook will guide you through:  

1.   **package installation and testing**: microbial association network requires the installation of several packages with potential problems which may be platform-specific and difficult to diagnose and solve; the approach used here is platform independnt but it can easily be transferred to other platforms
2.   **network inference using a variety of methods**: for a review on approaches and methods for the inference of Microbial Association Networks
3.   **analysis of network properties**: network statistics with special emphasis on those related to structure and stability, including node degree distribution and tests for small-world scale-free structure
4.   **analysis of nodes and edges properties**: analysis of centralities and communities detection, with special emphasis on aspects related to keystone taxa

To ensure reproducibility and reduce execution time and memory usage I will focus on small data sets included in the packages we are going to use.
**Please be patient and wait until each cells completes its processes**

# The packages you are going to use (and why you need them).  

Using a notebook is quite different from a local session with R, where, once you have installed your packages, you can safely find them in your library (until a new minor version is installed: R only preserves libraries between patches i.e. 4.4.1 to 4.4.2). On the other hand a major advantage of this notebook is that it is reproducible. Many of the packages we are going to install come with many dependencies. I have chosen to explicitly install some packages even if they would be installed and loaded anyway by other packages. I am now listing the main packages we are going to use and why they are needed:

1.   Packages for data wrangling and microbiome analysis
     1. [tidyverse](https://tidyverse.org/): includes most of the packages you need for data manipulation
     2. [phyloseq](https://joey711.github.io/phyloseq/): for handling and exploring microbiome data as `phyloseq` objects; an alternative would be `mia` combined with `miaverse`: both are part of the [miaverse](https://microbiome.github.io/)
2.   Packages for MAN analysis and network analysis:
     1. [igraph](https://r.igraph.org/): is the package for network analysis in R, but its syntax can be a bit obscure
     2. [tidygraph](https://tidygraph.data-imaginist.com/reference/index.html) combined with [ggraph](https://ggraph.data-imaginist.com/) is an excellent alternative, in terms of tidy data and integration with tidyverse tools: unfortunately the documentation is poor
     3. (NetCoMi)[https://netcomi.de/]: IMHO is currently the best package for MAN inference and analysis in R. It provides excellent functions for network inference, calculation of network, node and edge statistics, network comparison and provides access to a very vast array of network inference methods. However, it uses R base graphics, which might be annoying and its functions do have quite a lot of options

## CoLab vs Quarto.  

I have designed the notebook so that it can be used in Google CoLab (with optional backup of the library in Google Drive) and in Quarto. Just set the `context` variable accordingly in the following block.

In [ ]:
context <- "CoLab" # set to "Quarto" if you want to use this with RStudio or Positron IDEs

## 0. Blast it all! Starting with a clean slate for package installation in R.  
If you want to start with a clean slate run the following cell: this procedure might be useful for debugging. As a safety measure you have to set `blast_me <- TRUE` first.    


In [ ]:
# --- PART 0: Clean slate (use with extreme caution) ---
# This cell deletes ALL packages installed in this session.
# To activate, you must set BOTH flags to TRUE.

blast_me         <- FALSE   # <- change to TRUE only if you are sure
i_am_really_sure <- FALSE   # <- also change this to TRUE

if (blast_me && i_am_really_sure && context == "CoLab") {
  message("⚠️  WARNING: Deleting all packages from /content/R_library_local ...")
  system("rm -rf /content/R_library_local/*")
  # Also reset the needs_backup and local_lib state
  needs_backup <- FALSE
  if (dir.exists("/content/R_library_local"))
    message(">>> Done. Run from Part 1 to reinstall.")
} else if (blast_me && !i_am_really_sure) {
  message(">>> blast_me is TRUE but i_am_really_sure is FALSE. Nothing was deleted.")
  message(">>> Set both flags to TRUE if you genuinely want to reset.")
} else {
  message(">>> Clean slate skipped. Nothing was deleted.")
}

## 1. (Optionally) Restoring from GoogleDrive.  
Sadly, to run NetCoMi you install several packages as dependencies. While this is fine when you run R locally, CoLab creates a new session each time and does not store your R locally. I  have designed this section to optionally save a backup of the library on your Google Drive. You can set the name of the Drive backup by setting `drive_zip_path <- "R_library_colab.zip"` or whatever you called the library last time you ran the script. You can entirely skip this by setting `install_on_drive <- F`.  
This section only opertates when the `context` is "CoLab".  

In [ ]:
1# --- PART 1: Automated Restore (Optimized) ---

# A small helper for visible status messages
colab_msg <- function(text, type = c("info", "success", "warning", "working")) {
  type <- match.arg(type)
  prefix <- switch(type,
    info    = "ℹ️  ",
    success = "✅  ",
    warning = "⚠️  ",
    working = "⏳  "
  )
  cat(paste0("\n", strrep("─", 50), "\n",
             prefix, toupper(type), ": ", text,
             "\n", strrep("─", 50), "\n\n"))
}

# Initialize context if not already set (e.g., if this is the first cell run)
if (!exists("context")) {
  context <- "CoLab" # Default to CoLab if not explicitly set elsewhere
  colab_msg("Context variable not found, defaulting to 'CoLab'.", "warning")
}

# Initialize variables based on context
if (context == "CoLab") {
  install_on_drive <- TRUE
  needs_backup     <- FALSE # Will be set to TRUE later if packages are installed
  local_lib <- "/content/R_library_local"
  if (!dir.exists(local_lib)) dir.create(local_lib)
  .libPaths(c(local_lib, .libPaths()))
  drive_zip_path <- "R_library_colab.zip"

  colab_msg("Running in Google Colab environment. Packages will be installed to a local Colab library.", "info")

  if (install_on_drive) {
    if (!require("googledrive", quietly = TRUE)) {
      colab_msg("Installing 'googledrive' for Google Drive interaction.", "working")
      install.packages("googledrive", lib = local_lib, repos = "https://cloud.r-project.org/")
      library(googledrive) # Load after installation
    } else {
      library(googledrive) # Load if already installed
    }

    # OOB auth for browser-based R sessions
    options(gargle_oob_default = TRUE)
    colab_msg("Please complete the Google Drive authentication in the popup.", "info")
    drive_auth()

    # Server-side exact name match (reliable alternative to pattern=)
    colab_msg("Looking for library backup on Google Drive...", "working")
    .drive_file <- drive_find(
      q     = paste0("name = '", drive_zip_path, "'"),
      n_max = 1
    )

    if (nrow(.drive_file) > 0) {
      colab_msg("Backup found! Downloading from Google Drive.\nThis may take a few minutes — please wait.", "working")
      drive_download(.drive_file, path = "lib_restore.zip", overwrite = TRUE)

      colab_msg("Extracting library — please wait...", "working")
      system("unzip -oq lib_restore.zip -d /content")
      file.remove("lib_restore.zip")

      colab_msg("Library restored successfully! Proceeding to package checks.", "success")
    } else {
      colab_msg(paste0("No backup found on Google Drive ('", drive_zip_path, "').\n",
                       "Packages will be installed from scratch — this will take longer."), "warning")
    }
  } else {
    colab_msg("Google Drive backup and restore skipped by user setting (`install_on_drive = FALSE`).", "info")
  }
} else { # context == "Quarto" or any other local environment
  install_on_drive <- FALSE # No Drive operations
  needs_backup     <- FALSE # No backup needed for local install
  local_lib        <- NULL # R will use default library paths
  drive_zip_path   <- NULL # Not applicable

  colab_msg("Running in local (Quarto/RStudio) environment. Packages will be installed to user's default library.", "info")
}

In [ ]:
# --- GUARD: Shared state initialization ---
# Run this if you are starting from Part 2 onwards without running Part 1.
# It is safe to run even if Part 1 has already been run.

# Ensure context is defined, default to "CoLab" if not
if (!exists("context")) {
  context <- "CoLab"
  message(">>> context was not initialized, defaulting to 'CoLab' in GUARD.")
}

if (context == "CoLab") {
  if (!exists("local_lib")) {
    local_lib <- "/content/R_library_local"
    if (!dir.exists(local_lib)) dir.create(local_lib)
    .libPaths(c(local_lib, .libPaths()))
    message(">>> local_lib initialized for CoLab in GUARD.")
  }

  if (!exists("needs_backup")) {
    needs_backup <- FALSE
    message(">>> needs_backup initialized for CoLab in GUARD.")
  }

  if (!exists("install_on_drive")) {
    install_on_drive <- TRUE
    message(">>> install_on_drive initialized (default: TRUE) for CoLab in GUARD.")
  }

  if (!exists("drive_zip_path")) {
    drive_zip_path <- "R_library_colab.zip"
    message(">>> drive_zip_path initialized (default) for CoLab in GUARD.")
  }
} else { # context is Quarto or local
  if (!exists("local_lib")) {
    local_lib <- NULL # R will use default library paths
    message(">>> local_lib will use R's default for local context in GUARD.")
  }
  if (!exists("needs_backup")) {
    needs_backup <- FALSE
    message(">>> needs_backup initialized for local context in GUARD.")
  }
  if (!exists("install_on_drive")) {
    install_on_drive <- FALSE
    message(">>> install_on_drive initialized (default: FALSE) for local context in GUARD.")
  }
  if (!exists("drive_zip_path")) {
    drive_zip_path <- NULL
    message(">>> drive_zip_path not applicable for local context in GUARD.")
  }
}

## 2. Installing CRAN and BioConductor packages.  
This section installs several packages which do not need compilation (will go on Drive if `install_on_drive == T`).

In [ ]:
# --- PART 2: System & Engine Setup ---

# System libraries always needed (not saved in Drive backup)
if (context == "CoLab") {
  colab_msg("Installing system libraries (always required, not cached)...", "working")
  system(
    "sudo apt-get update -qq && sudo apt-get install -y -qq libglpk-dev libgmp3-dev libxml2-dev libgsl-dev libfftw3-dev r-base-dev",
    ignore.stdout = TRUE, ignore.stderr = TRUE
  )
  colab_msg("System libraries ready.", "success")
} else {
  colab_msg("Skipping system library installation as context is not 'CoLab'.", "info")
}

# 1. Define required packages
.cran_packages <- c(
  "devtools", "Rcpp", "RcppArmadillo", "Matrix", "huge", "MASS",
  "glmnet", "igraph", "tidyverse", "tictoc", "beepr", "logr",
  "VennDiagram", "lobstr", "tidygraph", "ggraph", "cowplot", "RColorBrewer",
  "mediation", "pulsar", "VGAM", "rootSolve", "BiocManager", "graphlayouts",
  "LaplacesDemon", "permute", "randomcoloR"
)

.bioc_packages <- c(
  "phyloseq", "DESeq2", "DirichletMultinomial", "DECIPHER", "limma"
)

# GitHub packages (always latest)
.github_packages <- list(
  SpiecEasi = "zdk123/SpiecEasi",
  mixedCCA  = "irinagain/mixedCCA",
  SPRING     = "GraceYoon/SPRING",
  NetCoMi   = "stefpeschel/NetCoMi"
)

# Combine CRAN + Bioc for pak
all_reqs <- c(.cran_packages, paste0("bioc::", .bioc_packages))

# Install pak itself if missing
# Use local_lib for Colab, or NULL for default library path in local environments
if (!require("pak", quietly = TRUE, lib.loc = local_lib)) {
  colab_msg("Installing pak package manager...", "working")
  install.packages("pak", lib = local_lib, dependencies = TRUE,
                   INSTALL_opts = c("--no-html", "--no-help"), repos = "https://cloud.r-project.org/")
  if (!require("pak", quietly = TRUE, lib.loc = local_lib)) {
    colab_msg("Failed to install pak. Please check system dependencies.", "warning")
    stop("pak installation failed.")
  }
}

# 2. Check/install CRAN and Bioconductor packages
# Skip if library was successfully restored from Drive and context is CoLab
library_was_restored <- exists("needs_backup") && !needs_backup &&
                        !is.null(local_lib) && length(list.files(local_lib)) > 0 && context == "CoLab"

if (library_was_restored) {
  colab_msg("Library restored from Drive — skipping initial CRAN/Bioc installation.", "info")
  install_plan <- NULL
  # Explicitly check for and install any missing CRAN packages if they're not in the restored library
  missing_cran_pkgs <- .cran_packages[sapply(.cran_packages, function(pkg) !requireNamespace(pkg, lib.loc = local_lib, quietly = TRUE))]
  if (length(missing_cran_pkgs) > 0) {
    colab_msg(paste0("Installing missing CRAN packages: ", paste(missing_cran_pkgs, collapse = ", "), "..."), "working")
    pak::pkg_install(missing_cran_pkgs, lib = local_lib)
    # After installing, check if any of these installations should trigger a backup
    # This part of the logic might need further refinement based on pak's return for individual installs
    if (context == "CoLab") {
      needs_backup <- TRUE
    }
    colab_msg("Missing CRAN packages installed.", "success")
  } else {
    colab_msg("All CRAN packages are present in the restored library.", "success")
  }

  missing_bioc_pkgs <- .bioc_packages[sapply(.bioc_packages, function(pkg) !requireNamespace(pkg, lib.loc = local_lib, quietly = TRUE))]
  if (length(missing_bioc_pkgs) > 0) {
    colab_msg(paste0("Installing missing Bioconductor packages: ", paste(missing_bioc_pkgs, collapse = ", "), "..."), "working")
    pak::pkg_install(paste0("bioc::", missing_bioc_pkgs), lib = local_lib)
    if (context == "CoLab") {
      needs_backup <- TRUE
    }
    colab_msg("Missing Bioconductor packages installed.", "success")
  } else {
    colab_msg("All Bioconductor packages are present in the restored library.", "success")
  }


} else {
  colab_msg("Installing CRAN and Bioconductor packages.\nThis will take a while on first run — please wait.", "working")
  install_plan <- pak::pkg_install(all_reqs, lib = local_lib)

  # Determine if backup is needed based on what changed
  change_types  <- c("install", "update", "build", "downgrade")
  changed_pkgs  <- install_plan$package[install_plan$type %in% change_types]

  if (length(changed_pkgs) > 0) {
    if (length(changed_pkgs) == 1 && changed_pkgs == "BiocManager") {
      colab_msg("Only BiocManager was updated — skipping backup.", "info")
    } else {
      colab_msg(paste0("Packages updated: ",
                       paste(changed_pkgs, collapse = ", ")), "info")
      if (context == "CoLab") { # Only set needs_backup for CoLab
        needs_backup <- TRUE
      }
    }
  } else {
    colab_msg("All packages are up to date.", "success")
  }
}

# --- Helper Function for GitHub Packages ---
check_github_package <- function(repo_path, pkg_name, force_reinstall = FALSE) {

  # --- Guard: ensure all globals exist regardless of cell execution order ---
  if (!exists("colab_msg", envir = .GlobalEnv, mode = "function")) {
    # Minimal fallback if Part 1 was never run
    colab_msg <<- function(text, type = c("info", "success", "warning", "working")) {
      type <- match.arg(type)
      prefix <- switch(type,
        info    = "ℹ️  ",
        success = "✅  ",
        warning = "⚠️  ",
        working = "⏳  "
      )
      cat(paste0(
        "\n", strrep("─", 50), "\n",
        prefix, toupper(type), ": ", text,
        "\n", strrep("─", 50), "\n\n"
      ))
    }
    colab_msg("colab_msg() was not initialized — defined now.", "warning")
  }

  if (!exists("context", envir = .GlobalEnv)) {
    assign("context", "CoLab", envir = .GlobalEnv)
    colab_msg("context was not initialized — defaulting to 'CoLab' in check_github_package guard.", "warning")
  }

  if (!exists("local_lib", envir = .GlobalEnv)) {
    if (get("context", envir = .GlobalEnv) == "CoLab") {
      assign("local_lib", "/content/R_library_local", envir = .GlobalEnv)
      if (!dir.exists(local_lib)) dir.create(local_lib)
      .libPaths(c(local_lib, .libPaths()))
      colab_msg("local_lib was not initialized — set to default for CoLab in check_github_package guard.", "warning")
    } else {
      assign("local_lib", NULL, envir = .GlobalEnv)
      colab_msg("local_lib not explicitly set for local context in check_github_package guard.", "info")
    }
  }

  if (!exists("needs_backup", envir = .GlobalEnv)) {
    assign("needs_backup", FALSE, envir = .GlobalEnv)
    colab_msg("needs_backup was not initialized — set to FALSE in check_github_package guard.", "warning")
  }

  if (!exists("library_was_restored", envir = .GlobalEnv)) {
    # Infer conservatively: if local_lib has files and context is CoLab, assume a restore happened
    restored <- get("context", envir = .GlobalEnv) == "CoLab" &&
                !is.null(get("local_lib", envir = .GlobalEnv)) &&
                length(list.files(get("local_lib", envir = .GlobalEnv))) > 0
    assign("library_was_restored", restored, envir = .GlobalEnv)
    colab_msg(paste0(
      "library_was_restored was not initialized — inferred as ",
      restored, " in check_github_package guard."
    ), "warning")
  }

  if (!exists("pak", envir = .GlobalEnv, mode = "function") &&
      !requireNamespace("pak", quietly = TRUE, lib.loc = local_lib)) { # Check all lib trees if local_lib is NULL
    colab_msg("pak is not available — attempting to install...", "working")
    install.packages("pak", lib = local_lib, dependencies = TRUE,
                     INSTALL_opts = c("--no-html", "--no-help"), repos = "https://cloud.r-project.org/")
    if (!requireNamespace("pak", quietly = TRUE, lib.loc = local_lib)) {
      colab_msg(paste0(
        "pak could not be installed. Cannot proceed with ",
        pkg_name, "."
      ), "warning")
      return(invisible(FALSE))
    }
  }

  # --- Main logic ---
  colab_msg(paste0("Checking: ", pkg_name), "working")

  # If library was restored (which implies context == "CoLab"), check if package is already present
  # Skip installation ONLY IF NOT force_reinstall and package is already present and loadable
  if (!force_reinstall && library_was_restored &&
      requireNamespace(pkg_name, lib.loc = local_lib, quietly = TRUE)) {
    message(paste0(
      ">>> [SKIP] ", pkg_name,
      " already present in restored library."
    ))
    pkg_loaded <- require(pkg_name, character.only = TRUE,
                          lib.loc = local_lib, quietly = TRUE)
    if (pkg_loaded) {
      colab_msg(paste0(pkg_name, " loaded from restored library."), "success")
    } else {
      colab_msg(paste0(
        pkg_name, " is in the library but failed to load.\n",
        "This may indicate a missing system library.\n",
        "Try running the blast_me cell and starting fresh." # Or force_reinstall=TRUE
      ), "warning")
    }
    return(invisible(pkg_loaded))
  }

  # Otherwise install (or update) from GitHub
  colab_msg(paste0(
    "Installing ", pkg_name, " from GitHub — please wait.\n",
    "This may take several minutes."
  ), "working")

  install_ok <- tryCatch({
    install_status <- pak::pkg_install(repo_path, lib = local_lib, ask = FALSE)
    change_types   <- c("install", "update", "build", "downgrade")
    if (any(install_status$type %in% change_types)) {
      message(paste0(">>> [DISK WRITE] ", pkg_name, " triggered a library change."))
      # Only set needs_backup to TRUE if in CoLab context
      if (get("context", envir = .GlobalEnv) == "CoLab") {
        assign("needs_backup", TRUE, envir = .GlobalEnv)
      }
    } else {
      message(paste0(">>> [CURRENT] ", pkg_name, " is already at the latest version."))
    }
    TRUE
  }, error = function(e) {
    colab_msg(paste0(
      "Installation of ", pkg_name, " failed:\n", e$message, "\n",
      "Check your internet connection or GitHub availability."
    ), "warning")
    FALSE
  })

  if (!install_ok) return(invisible(FALSE))

  # Load and report
  pkg_loaded <- require(pkg_name, character.only = TRUE,
                        lib.loc = local_lib, quietly = TRUE)
  if (pkg_loaded) {
    colab_msg(paste0(pkg_name, " installed and loaded successfully."), "success")
  } else {
    colab_msg(paste0(
      pkg_name, " was installed but could not be loaded.\n",
      "A system library may be missing — check the apt-get cell\n",
      "in Part 2 and make sure it completed successfully."
    ), "warning")
  }

  return(invisible(pkg_loaded))
}


### 3. Installing and testing SpiecEasi.  
I will now start installing GitHub packages, one by one, if they are not available in the restored library. I am installing [SpiecEasi](https://github.com/zdk123/SpiecEasi) first, using `pak`.

In [ ]:
# --- PART 3: The Clean Approach to SpiecEasi install ---

check_github_package(.github_packages[["SpiecEasi"]], "SpiecEasi")

---
**Testing SpiecEasi**

I will now quickly test the use of `SpiecEasi` package. SPIEC-EASI algorithm is one of the best performing MAN inference algorithms (Kurtz et al., 2015; Tackmann et al., 2019; Ghaeli et al., 2025). SpiecEasi can also be used with NetCoMi. Here, I am going to quickly test inference with the Meinshausen-Buhlmann algorithm, which is arguably faster and more robust especially with larger datasets than the glasso algorithm. The data set I am going to use is provided in the package and it is part of the American Gut Project.  
The parameters of the algorithm are chosen for speed, rather than for precision: `nlambda` should be in the range of 20-30, 50 offers higher precision but slows computation exponentially.  
Note also that you should check the [SpiecEasi documentation](https://github.com/zdk123/SpiecEasi) (although, in my opinion, it tends to be a bit cryptic).
The `spiec.easi()` function returns a `spiec.easi` object, a list.


In [ ]:
require(SpiecEasi)
require(igraph)

# Load amgut
data("amgut1.filt")
class(amgut1.filt)

# check if it has been loaded in memory
ls()

# use amgut for SpiecEasi inference
se.out <- spiec.easi(amgut1.filt, method='mb',
                     lambda.min.ratio=1e-2,
                     nlambda=10)
# when working locally consider using

if(context == "Quarto" && !exists("use_cores")){
  require(parallel)
  use_cores <- max(1,(detectCores()-1))
  if (is.null(use_cores) || is.na(use_cores)) use_cores <- 1L
  se.out <- spiec.easi(amgut2.filt.phy,
                     method='mb', lambda.min.ratio=1e-2,
                     nlambda=20, pulsar.params=list(rep.num=20, ncores=use_cores))
}


Reasonable indications for parameters choice are:
```
se.out <- spiec.easi(
  my_phyloseq_object,
  method = "mb",                 # Or "glasso" based on your taxon count
  nlambda = 30,                  # Smooth, balanced path resolution
  lambda.min.ratio = 1e-2,       # Standard path depth
  sel.criterion = "stars",       # Use StARS stability selection
  pulsar.params = list(
    rep.num = 20,                # 20 random subsamples for variance estimation
    thresh = 0.05,               # Conservative stability threshold
    ncores = 4                   # Parallelize across available CPU cores
  )
)
```
The object we need is now `se.out`; let's check and then use `igraph` to plot. Note how we need to convert the adjiacency matrix to a igraph object. We will see later that `ggraph` offers better and more intuitive plotting options for networks.

In [ ]:
class(se.out)
ig.mb     <-  adj2igraph(getRefit(se.out)) #converts to igraph
## set size of vertex proportional to clr-mean
vsize    <- rowMeans(clr(amgut1.filt, 1))+6
am.coord <- igraph::layout_with_fr(ig.mb) # applies a force based Fruchterman-Reingold layout
options(repr.plot.width = 10, repr.plot.height = 10)
par(mfrow=c(1,1))

# plot using igraph
plot(ig.mb,
     layout = am.coord,
     vertex.size = vsize,
     vertex.label = NA,
     main = "Microbiome Network: MB Method")



---
**Installing heavy GitHub packages**

OK, now the heavy part, which is installing NetCoMi (its dependencies first)
from GitHub, again if needed. I am splitting this in multiple blocks to isolate the problems. **Note that 5.3 can take forever (almost...)**

In [ ]:
# --- PART 5.1: mixedCCA install ---

check_github_package(.github_packages[["mixedCCA"]], "mixedCCA")


In [ ]:
# --- PART 5.2: SPRING install ---


check_github_package(.github_packages[["SPRING"]], "SPRING")


In [ ]:
# --- PART 5.3: The Clean Approach to NetCoMi install ---

check_github_package(.github_packages[["NetCoMi"]], "NetCoMi")


---
## **Testing NetCoMi for network inference and analysis**

We are finally there: I am now testing NetCoMi with code from https://netcomi.de/articles/netcomi

In [ ]:
# --- PART 6: Test if NetCoMi is loaded ---

require(NetCoMi)

help(netConstruct)

## **Saving the library**.
Installing and loading the packages was exhausting. I will now conditionally save the library to my drive to make the use of the notebook faster next time it is opened. **Note**: I have left a back door to reset needs_backup, just set `reset_needs_backup to <-T`.

In [ ]:
# --- PART 7: Robust Sync to Drive ---

# Force a backup regardless of needs_backup by setting this to TRUE
reset_needs_backup <- FALSE
if (reset_needs_backup) needs_backup <- TRUE

if (exists("needs_backup")) {
  cat("\nneeds_backup flag is", needs_backup, "\n")
} else {
  colab_msg("needs_backup not defined — skipping backup.", "info")
}

if (install_on_drive && exists("needs_backup") && needs_backup) {

  colab_msg("Compressing library — this may take a moment...", "working")
  old_wd <- getwd()
  setwd("/content")
  system("zip -r lib_backup.zip R_library_local")

  if (file.exists("lib_backup.zip")) {

    colab_msg("Uploading library to Google Drive.\nDo not close this tab.", "working")

    # Server-side exact name match (consistent with Part 1)
    .drive_file <- drive_find(
      q     = paste0("name = '", drive_zip_path, "'"),
      n_max = 1
    )

    if (nrow(.drive_file) > 0) {
      googledrive::drive_update(.drive_file, "lib_backup.zip")
      colab_msg("Google Drive backup updated successfully.", "success")
    } else {
      googledrive::drive_upload("lib_backup.zip", name = drive_zip_path)
      colab_msg("New Google Drive backup created successfully.", "success")
    }

    file.remove("lib_backup.zip")

  } else {
    colab_msg("Zip file was not created — check available disk space.", "warning")
    stop("Aborting: backup zip not found.")
  }

  needs_backup <- FALSE
  setwd(old_wd)

} else {
  colab_msg("No changes detected — Google Drive backup not needed.", "info")
}

# Using NetCoMi for network inference and analysis.  

MAN inference and analysis requires several steps each of which needs to be carefully planned. Please refer to the literature cited in this notebook for further details. An outline of the steps we are going to carry out (often implicitly, filtering and standardization is embedded in the `netConstruct()` function) is depicted quite nicely in this figure
![Fig 1 from Peschel et al. 2020.](https://drive.google.com/uc?export=view&id=1-76cyFQ0GpTlRjU6035j-wIfXBXtXlm4)

## Preparing your data for network inference with `netConstruct()`.  

`netConstruct()` is the primary function used by NetCoMi for network inference. It is incredibly powerful but also difficult to use, because of the very many options it has. Read carefully [NetCoMi documentation](https://netcomi.de/articles/netcomi) and [function reference](https://netcomi.de/reference/).  
The options can be consulted [here](https://netcomi.de/reference/netconstruct).

Data fed to the function can be a numeric matrix (samples in rows, OTUs/ASVs in columns) or an object of the classes: [**phyloseq**](https://joey711.github.io/phyloseq/import-data.html), SummarizedExperiment, [**TreeSummarizedExperiment**](https://microbiome.github.io/OMA/docs/devel/pages/convert.html. All require "count" as dataType. The data can also be an association or dissimilarity matrix (dataType must be set accordingly): see [here](https://netcomi.de/articles/cross_domain_networks) for an example of use with an association matrix.
The function has two slots; only the first one is used unless you want to perform network comparison.
The two options in bold are the one I prefer because they are a simple way to pass both data and metadata (including taxonomic metadata).  



### Example 1. Using the amgut data from NetCoMi.  

NetCoMi and phyloseq come with several data sets, in different formats. We will use the amgut data set.

In [ ]:
# --- PART 8.1: (re)loading NetCoMi data ---

library(NetCoMi)
library(phyloseq)
data(amgut1.filt) # ASV data count
data(amgut2.filt.phy) # phyloseq
# uncomment and execute these commands if you want to have a look to the data
# look for accessor functions in the phyloseq help https://joey711.github.io/phyloseq/preprocess.html
# amgut1.filt
# help(amgut1.filt)
# str(amgut2.filt.phy)


Both of these datasets contain ASVs. One of the first choices you have to make is the level of aggregation you want to use in the analysis:  

1.   **ASV**: technically the lowest and the most agnostic level (you only attach taxonomic assignment ex post), somebody interprets them as real entities, something between species and strain. **Pro**: theoretically, real interaction can be between strains within species, ASVs require less assumptions; **Cons**: a single species may have several RNA operons, which may result in different ASVs; the count tables can be extremely sparse; in my experience, you end up having several indirect associations between ASVs belonging to the same species (which can be collapsed in a single node in a network) and parent-child relationships (an association between an ASVs assigned to a higher taxon, i.e. _Lactobacillus_ or _Lactobacillaceae_ and a lower taxon, i.e. _Lactobacillus delbrueckii_ which are difficult to interpret
2.   **Species**: you can technically agglomerate at the species level, using for example tax_glom() function in phyloseq. **Pro**: more interpretable than ASVs; **Cons**: species assignment can be unreliable with short amplicons (V4) for bacteria, although many sequences can get species assignment with Fungi using ITS as a target; may result in duplicate lables; read carefully tax_glom() documentation...
3.  **Genus** or above: again here you can use tax_glom() and this also removes ASVs which have taxonomic assignments above the genus level. **Pros** the tables are much less sparse and you also perform some filtering; acknowledges the limitations of taxonomic assignmet with amplicon targeted metagenomics; **Cons**: technically, different species within the same genus may have different ecologies and the data can be less interpretable; may get duplicate taxa names

We will follow the NetComi tutorial and assume that some filtering has already been performed. Have a look at the phyloseq tutorial for some ideas on filtering: good ideas could be removing taxa lacking assignment above a given taxonomic level (done by tax_glom()), removing Chloroplasts and Mitochondria, etc.
Further taxa filtering can be done using prevalence and abundance filters. Have a look at [this function](https://github.com/ep142/FoodMicrobionet/blob/master/miscellaneous/filter_my_taxa_function_1_4.R) for other ideas.
As an alternative, you can set filtering parameters in the netConstruct() function. Look at the [reference](https://netcomi.de/reference/netconstruct) under "preprocessing".
Let's perform taxonomic agglometarion and taxa renaming on the amgut.2.filt.phy object.
Note that [renameTaxa](https://netcomi.de/reference/renametaxa) is a very convenient NetCoMi function,

In [ ]:
# --- Part 8.2 Taxa agglomeration ---

require(NetCoMi)
require(phyloseq)

# Load data sets
data("amgut1.filt") # ASV count matrix
data("amgut2.filt.phy") # phyloseq objext

# Agglomerate to genus level
amgut_genus <- tax_glom(amgut2.filt.phy, taxrank = "Rank6")
taxtab <- tax_table(amgut_genus)
cat("\nBefore renaming...\n")
head(taxtab)
cat("\nOnly Eubacterium...\n")
taxtab[taxtab[, "Rank6"] == "g__Eubacterium", ]
# Rename taxonomic table and make Rank6 (genus) unique
amgut_genus_renamed <- renameTaxa(amgut_genus,
                                  pat = "<name>",
                                  substPat = "<name>_<subst_name>(<subst_R>)",
                                  numDupli = "Rank6")

taxtab_2 <- tax_table(amgut_genus_renamed)
cat("\nAfter renaming...\n")
head(taxtab_2)
# unless you want the OTU names asnode names in the network you also need to rename the taxa...
amgut_genus_renamed_2 <- amgut_genus_renamed
taxa_names(amgut_genus_renamed_2) <- unname(taxtab_2[,6])
head(taxa_names(amgut_genus_renamed_2))



Note that nodes, by default, will have the rowname as a name, unless you use something different. Names must be unique.

## Handling zeros.  

Count tables are sparse (sometimes very sparse). 0s can prevent some transformations (like Centered Log Ratio) and their meaning is ambiguous. Excess 0s can also inclate correlation measures. Check out these references if you want to know more:

- Chan, L.S., Li, G., 2024. Zero is not absence: censoring-based differential abundance analysis for microbiome data. Bioinform. (Oxf., Engl.) 40, btae071. https://doi.org/10.1093/bioinformatics/btae071.
- Kaul, A., Mandal, S., Davidov, O., Peddada, S.D., 2017. Analysis of Microbiome Data in the Presence of Excess Zeros. Front. Microbiol. 8, 2114. https://doi.org/10.3389/fmicb.2017.02114
- Silverman, J.D., Roche, K., Mukherjee, S., David, L.A., 2020. Naught all zeros in sequence count data are the same. Comput. Struct. Biotechnol. J. 18, 2789–2798. https://doi.org/10.1016/j.csbj.2020.09.014

## Transformations.
Transformations are also needed for some method (notably, clr should be used for correlation based methods). Read the NetCoMi paper for this. Both Zero handling and transformations are explained very well in the supplementary table S1 of the Pechel et al. (2020) paper.

## Setting the options.  
If you plan to combine estimation with several methods it may be convenient to create a list for parameters to be passed to `netConstruct()`.

In [ ]:
# --- PART 8.3: setting the parameters ---

# methods
# spieceasi_mb SpiecEasi with MB (Meinshausen-Bühlmann neighbourhood selection)
# spieceasi_glasso SpiecEasi with glasso
# spearman_fdr uses Spearman correlations with false discovery rate, fast
# sparcc_thresh uses sparcc as a method with no permutation, fast
# sparcc_perm uses sparcc with permutations, possibly slow
# ccrepe, possibly slow
# spring, possibly very slow
# the two spiec-easi might be quite slow

inf_methods <- c("spieceasi_mb") # or more, you can use this with map2...

# set alpha for thresholds and sparsigication
p <- list(
  network_alpha = 0.05
  )

# redundant
if(context == "Quarto" && !exists("use_cores")){
  require(parallel)
  use_cores <- max(1,(detectCores()-1))
  if (is.null(use_cores) || is.na(use_cores)) use_cores <- 1L
}

# the list of parameters to be passed in `netConstruct()` for each of the methods;
# there must be one entry for each of the parameters in the list
# Shared defaults
my_seed <- 1234 # (for reproducibility purposes)
net_defaults <- list(
    dissFunc  = "signed",
    verbose   = 3,
    dataType  = "counts",
    seed = my_seed
)
# 3 is very verbose, consider using 1

# another safeguard to make this work in multiple contexts
cores_ok <- exists("use_cores", inherits = TRUE) &&
            !is.null(use_cores) &&
            !anyNA(use_cores)

if (cores_ok) net_defaults$cores <- use_cores

# Method-specific configurations
inf_methods_param <- list(

    # ── SpiecEasi with MB (Meinshausen-Bühlmann neighbourhood selection) ──────
    # sparsification is handled internally by SpiecEasi (StARS stability)
    # sparsMethod = "none" because SpiecEasi produces a sparse matrix directly
    spieceasi_mb = modifyList(net_defaults, list(
        measure      = "spieceasi",
        measurePar   = list(
            method           = "mb",
            lambda.min.ratio = 1e-2,
            nlambda          = 25,
            pulsar.params    = list(rep.num = 50),
            symBetaMode      = "ave"
        ),
        normMethod   = "none",     # SpiecEasi handles normalisation internally
        zeroMethod   = "none",     # SpiecEasi handles zeros internally
        sparsMethod  = "none"      # sparsification built into SpiecEasi
    )),
    # ── SpiecEasi with glasso (graphical lasso)
    spieceasi_glasso = modifyList(net_defaults, list(
      measure      = "spieceasi",
      measurePar   = list(
          method           = "glasso",
          lambda.min.ratio = 1e-2,      # same path width as your mb run — keeps the two comparable
          nlambda          = 25,
          pulsar.params    = list(
              rep.num = 50      # glasso's StARS subsampling is the expensive part — parallelize it explicitly
          ),
          symBetaMode      = "ave"      # largely a no-op for glasso (already symmetric), kept for consistency with spieceasi_mb
      ),
      normMethod   = "none",     # SpiecEasi handles normalisation internally
      zeroMethod   = "none",     # SpiecEasi handles zeros internally
      sparsMethod  = "none"      # StARS sparsification built into SpiecEasi
    )),

    # CCREPE
    # sparsification via FDR (adaptBH) on CCREPE's internal resampling p-values
    # normMethod = "fractions" since CCREPE expects relative abundances
    ccrepe = modifyList(net_defaults, list(
        measure      = "ccrepe",
        measurePar   = NULL,
        normMethod   = "fractions",
        zeroMethod   = "none",
        sparsMethod  = "t-test",   # uses CCREPE's internal p-values
        alpha        = as.numeric(p$network_alpha),
        adjust       = "adaptBH"
    )),

    # Spearman with FDR sparsification
    # simplest compositionally naive option; honest about limitation
    # normMethod = "clr" reduces compositionality bias for correlation methods
    spearman_fdr = modifyList(net_defaults, list(
        measure      = "spearman",
        measurePar   = NULL,
        normMethod   = "clr",
        zeroMethod   = "pseudoZO",  # add pseudocount to zeros only
        zeroPar      = list(pseudocount = 0.5),
        sparsMethod  = "t-test",
        alpha        = as.numeric(p$network_alpha),
        adjust       = "adaptBH"
    )),

    # SparCC with threshold sparsification (simplest, no permutations)
    # thresh = minimum |r| to retain an edge — acknowledged as arbitrary
    sparcc_thresh = modifyList(net_defaults, list(
        measure      = "sparcc",
        measurePar   = list(iter = 20, inner_iter = 10, th = 0.1),
        normMethod   = "none",      # SparCC handles compositionality internally
        zeroMethod   = "none",
        sparsMethod  = "threshold",    # simple threshold — no permutations
        thresh       = 0.3          # minimum |r| to retain edge
    )),

    # SparCC with permutation-based sparsification
    # computationally expensive but statistically grounded
    # nboot controls number of permutations for null distribution
    sparcc_perm = modifyList(net_defaults, list(
        measure      = "sparcc",
        measurePar   = list(iter = 20, inner_iter = 10, th = 0.1),
        normMethod   = "none",
        sparsMethod  = "bootstrap", # permutation-based — equivalent cost to CCREPE
        nboot        = 1000L,
        alpha        = as.numeric(p$network_alpha),
        adjust       = "adaptBH"
    )),

    spring = modifyList(net_defaults, list(
      zeroMethod = "none",         # Handled internally by SPRING
      normMethod = "none",         # Handled internally by SPRING
      sparsMethod = "none",        # StARS handles sparsification internally

      # SPRING tuning parameters
      measurePar = list(
        Rmethod = "approx",        # CRITICAL for speed (uses interpolation)
        nlambda = 50,              # Number of regularization penalties to test
        rep.num = 50,              # Number of StARS subsamples (use 100 for final pub)
        thresh = 0.05,             # StARS edge variability threshold
        quantitative = FALSE       # Set to TRUE ONLY for true absolute counts
      )
  ))
)

### Testing the methods.  

Now use amgut_genus_renamed to test inference with several methods. Note that in complex workflows `netContruct` may fail and throw an error. It is usually better to trap this using `tryCatch()`. The code below shows two alternatives:  


1.   using a single method chosen from a vector of methods
2.   using a loop (I suggest this option rather that trying to use `purrr`)



In [ ]:
# as a reminder

# spieceasi_mb SpiecEasi with MB (Meinshausen-Bühlmann neighbourhood selection)
# spieceasi_gl SpiecEasi with glasso
# spearman_fdr uses Spearman correlations with false discovery rate, fast
# sparcc_thresh uses sparcc as a method with no permutation, fast
# sparcc_perm uses sparcc with permutations, possibly slow
# ccrepe, possibly slow
# spring, possibly very slow
# the two spiec-easi might be quite slow


inf_methods <- c("spieceasi_mb", "spieceasi_glasso", "spearman_fdr",
                 "sparcc_thresh", "sparcc_perm")
# or more, you can use this with map2...

i<-1 # can be easily adapted to a loop
mymethod <- inf_methods[i]
selected_params <- inf_methods_param[[mymethod]]

# inference is carried out here using a user defined function
# you do not need to provide much detail, everything is taken care of in the options
# of course could be customized in the function call
infer_MAN <- function(myphyseq, meth_and_param){
  mnet_object <- tryCatch(
    do.call(NetCoMi::netConstruct,
            c(list(data = myphyseq), meth_and_param)),
    error = function(e){
      msg <- paste0(
        format(Sys.time(), "%Y-%m-%d %H:%M:%S"), "\n",
        "measure: ", meth_and_param$measure, "\n",
        "error:   ", conditionMessage(e), "\n",
        paste(rep("-", 60), collapse = ""), "\n"
      )
      message("infer_MAN failed for measure '", meth_and_param$measure,
              "': ", conditionMessage(e))
      structure(class = "try-error", .Data = conditionMessage(e))
    }
  )
  return(mnet_object)
}
physeq_to_use <- amgut_genus_renamed_2
MAN_inf_simple <- infer_MAN(myphyseq = physeq_to_use,
                             meth_and_param = selected_params)


In [ ]:
# three methods in a loop
inf_methods <- c("spieceasi_mb", "spearman_fdr", "sparcc_thresh")

MAN_inf_results <- vector("list", length = length(inf_methods))

for(i in seq_along(inf_methods)){
  infmethod <- inf_methods[[i]]
  infparam <- inf_methods_param[[infmethod]]
  MAN_inf_results[[i]] <- infer_MAN(myphyseq = physeq_to_use,
                                  meth_and_param = infparam)
  names(MAN_inf_results)[i] <- infmethod
}

str(MAN_inf_results[[1]])

As with many other R packages, the value returned by `NetCoMi` is a very complex list (which is not tidy at all!). If you plan to use only NetCoMi you need not to worry, but if you want to extract the objects it contains you should really read the [(https://netcomi.de/reference/netconstruct). Important items:

*   the edge list(s) which can be passed easily to `igraph` or `tidygraph` see below;
*   the adjacency matrix, containing the weights: useful for evaluating the structure of the network using null models with igraph (out of scope for this workshop), can also be passed to `igraph` or `tidygraph`



In [ ]:
# build igraph and tidygraph objects
# the spieceasi network
se_net <- MAN_inf_results[["spieceasi_mb"]]
class(se_net)
# Extract edgelist from your microNetProps object (e.g., netProps)
edgelist_se <- se_net$edgelist1[, c("v1", "v2", "adja")]
colnames(edgelist_se) <- c("from", "to", "weight")

# Convert directly to igraph
library(igraph)
igraph_se <- graph_from_data_frame(edgelist_se, directed = FALSE)
# Using the 'edgelist' data frame created above
library(tidygraph)
tg_se <- as_tbl_graph(edgelist_se, directed = FALSE)
# Extract the adjacency matrix from your NetCoMi object
adj_matrix <- se_net$adja1

# Convert the matrix directly into an igraph object
library(igraph)
igraph_se_adj <- graph_from_adjacency_matrix(adj_matrix, mode = "undirected", weighted = TRUE, diag = FALSE)


As an exercise try to understand the structure of these objects. Also try accessors and look for help:

In [ ]:
summary(se_net)

## Converting to tidygraphs.  

When working with several objects in an automated workflow it might be nice to have a function with error trapping. Look at the following block, which takes a list with multiple microNet objects and turns it in a list with `tidygraphs`; I am using purrr: think how more complex, nested lists, could be handled with a loop or `imap`.

In [ ]:
# converting microNet in tidygraph objects --------------------------------
# requires: igraph, tidygraph, dplyr, tibble
require(purrr)
microNet_to_tidygraph <- function(micronet_obj,
                                  net_to_use = 1,
                                  add_names_to_edges = TRUE,
                                  use_asso_matrix = TRUE,
                                  fail_w_err = TRUE
) {
  # check required packages are installed
  required_pkgs <- c("igraph", "tidygraph", "dplyr", "tibble")
  missing_pkgs <- required_pkgs[!vapply(required_pkgs, requireNamespace, logical(1), quietly = TRUE)]
  if (length(missing_pkgs) > 0) {
    stop(
      "The following packages are required but not installed: ",
      paste(missing_pkgs, collapse = ", "),
      "\nInstall with: install.packages(c(",
      paste0('"', missing_pkgs, '"', collapse = ", "), "))"
    )
  }
  # check the class of the object
  if (!inherits(micronet_obj, "microNet")) {
    if (fail_w_err) {
      stop("this function only handles microNet objects\n")
    } else {
      warning("this function only handles microNet objects\n")
      return("not a microNet object")
    }
  } else {
    # get the adjacency matrix from microNet object
    if (net_to_use == 1) {
      adja_mat <- micronet_obj$adjaMat1
    } else {
      adja_mat <- micronet_obj$adjaMat2
    }
    # check if the adjacency matrix is NULL or not a matrix
    if (is.null(adja_mat) || !is.matrix(adja_mat)) {
      if (fail_w_err) {
        stop("cannot find the adjacency matrix\n")
      } else {
        warning("cannot find the adjacency matrix\n")
        return("adjacency matrix not found")
      }
    } else {
      # create igraph object from the adjacency matrix
      igraph_from_micronet <- igraph::graph_from_adjacency_matrix(
        adja_mat,
        mode = "lower",
        weighted = TRUE,
        diag = FALSE,
        add.colnames = NULL
      )
      # create tidygraph object; note this also has class igraph
      tidygraph_from_micronet <- tidygraph::as_tbl_graph(igraph_from_micronet)
      # check if there is at least one edge
      has_edges <- nrow(
        tidygraph_from_micronet |>
          tidygraph::activate(edges) |>
          tibble::as_tibble()
      ) >= 1
      if (has_edges) {
        # optionally add names to edges
        if (add_names_to_edges) {
          tidygraph_from_micronet <- tidygraph_from_micronet |>
            tidygraph::activate(edges) |>
            dplyr::mutate(from_name = .N()$name[from],
                          to_name = .N()$name[to])
        }
        if (use_asso_matrix) {
          # get the association matrix from microNet object
          if (net_to_use == 1) {
            asso_mat <- micronet_obj$assoMat1
          } else {
            asso_mat <- micronet_obj$assoMat2
          }
          # check if the association matrix is NULL or not a matrix
          if (is.null(asso_mat) || !is.matrix(asso_mat)) {
            if (fail_w_err) {
              stop("cannot find the association matrix\n")
            } else {
              warning("cannot find the association matrix\n")
              return(tidygraph_from_micronet)
            }
          } else {
            asso_graph <- igraph::graph_from_adjacency_matrix(
              asso_mat,
              mode = "lower",
              weighted = TRUE,
              diag = FALSE,
              add.colnames = NULL
            )
            tidy_asso_graph <- tidygraph::as_tbl_graph(asso_graph)
            edge_tidy_asso_graph <- tidy_asso_graph |>
              tidygraph::activate(edges) |>
              tibble::as_tibble() |>
              dplyr::rename(asso_est = weight) |>
              dplyr::mutate(asso_type = dplyr::if_else(asso_est > 0, "copres", "mut_ex"))
            # join the association measure to the main graph edges
            tidygraph_from_micronet <- tidygraph_from_micronet |>
              tidygraph::activate(edges) |>
              dplyr::left_join(edge_tidy_asso_graph, by = c("from", "to"))
            # use absolute association estimate as layout/display weight where available
            tidygraph_from_micronet <- tidygraph_from_micronet |>
              tidygraph::activate(edges) |>
              dplyr::mutate(weight = dplyr::if_else(!is.na(asso_est), abs(asso_est), abs(weight)))
            # add node label column
            tidygraph_from_micronet <- tidygraph_from_micronet |>
              tidygraph::activate(nodes) |>
              dplyr::mutate(nlabel = name)
          }
        }
        return(tidygraph_from_micronet)
      } else {
        warning("the network has 0 edges\n")
        return("no edges")
      }
    }
  }
}

tidygraph_list <- map(MAN_inf_results, microNet_to_tidygraph)
str(tidygraph_list)

## Calculating network statistics.  

This is the step which follows network inference and is required to generate an object which can be plotted. NetCoMi has a very powerful function, [`netAnalyze()`](https://netcomi.de/reference/netanalyze), which takes as an input a `microNet` object and returns a `microNetProps` object with network, node and edge statistics. As before, this is a very complex list with two slots if you started with the inference of two networks (something you need to do if you want to compare networks).  
Read carefully the [help](https://netcomi.de/reference/netanalyze), and the references I provided in [the introductory block](#https://colab.research.google.com/drive/1qn_IaKcOylegW_C_uUX5nbgCycePkpSo#scrollTo=HXhk5S66aDPq) the tutorial(https://netcomi.de/articles/netcomi) and the defaults before proceeding.
As an alternative, you can use the igraph object or the tidygraph object to calculate the statistics. Unfortunately, the documentation for both packages tends to be rather synthetic but this is nothing a good conversation with a LLM (Claude, not Gemini) can't fix.  
As before, you can use the `netAnalyze()` function directly but having a function wrapping it is useful when handling lists with multiple networks.

In [ ]:
require(NetCoMi)
# a single network, using defaults
MAN_inf_simple_wstats <- netAnalyze(MAN_inf_simple)
summary(MAN_inf_simple_wstats)


### Network statistics: food for thought.  

Try to understand the structure of the object which has been returned and remember what I have said the the lecture abuout network, node and edge stats (or simply, forget about thinking and ask a LLM):


*   which centralities are being returned?
*   in which way they are useful in the identification of "interesting" nodes (keystones, like hubs or bottlenecks?)
*   how can we tweak the parameters for hub detection?
*   which global network properties are useful to undestand the strucutre of the network?
    + is it large or small?  
    + is it fragmented?  
    + is it small world? scale free?
    + what is the significance of parameters like the modularity, the average clustering coefficient, the average path length, the density  

Note that formal tests exist to evaluate if a network is random or if it conforms to a specific network model but they are outside the scope of this workshop.

One important recent addition to recent versions of NetCoMi is the generation of a graphlet correlation matrix (GCM) which is an excellent indicator of network structure. The GCM is calculated by default, but it can be suppressed. A graphlet is a small, connected subgraph pattern (here, up to 4 nodes). For each graphlet shape, every node position within it is an orbit — a position that is topologically distinct from other positions in the same graphlet, in the sense that no automorphism (symmetry) of the graphlet maps it onto another position. Counting, for every node in your network, how many times it occupies each orbit position across all 2-to-4-node graphlets gives you a much richer local-structure fingerprint than degree alone. The 11 non-reduntant orbit counts answer to the following questions:

* Orbit 0 — degree: how many direct association partners does this taxon have? (Same as your existing degree centrality.)

* Orbits 2, 5, 7 — chain positions: how often is this taxon a "middle" or "link" node in a path of 3–4 taxa, i.e. how often does it sit between other taxa rather than at an endpoint?

* Orbits 8, 10, 11 — cycle positions: how often is this taxon part of a closed loop (triangle or 4-node cycle) of associations — i.e. local clustering/redundancy around that taxon, related to but more fine-grained than the clustering coefficient.

* Orbits 6, 9, 4, 1 — terminal/leaf positions: how often is this taxon a "dead end" — connected to the rest of the structure through just one neighbor in a larger graphlet, rather than being embedded in the middle of a denser local subgraph.

The GCM is the matrix of pairwise Spearman correlations between these orbit-count vectors across all nodes in the network.
For their interpretation refer to Przuli (2007) (Pržulj, N., 2007. Biological network comparison using graphlet degree distribution. Bioinformatics 23, e177–e183. https://doi.org/10.1093/bioinformatics/btl301) and  Yaveroğlu et al (2024) (Yaveroğlu, Ö.N., Malod-Dognin, N., Davis, D., Levnajic, Z., Janjic, V., Karapandza, R., Stojmirovic, A., Pržulj, N., 2014. Revealing the Hidden Language of Complex Networks. Sci. Rep. 4, 4547. https://doi.org/10.1038/srep04547) and to the NetCoMi documentation.
Look at from Pržulj et al. for heatmaps of GCM which are signature of important networks




### Doing it with a function.  

I will now show you how to do it programmatically. While you could directly use `imap` with `netAnalyze` on the `MAN_inf_results` list after retaining only objects inheriting the microNet class, the approach below allows to report any missing network.

In [ ]:
# using a function for extracting network statistics from a list
# calculate network stats -------------------------------------------------
# calculates net stats if possible, otherwise returns a try-error object.
# I am just using defaults for most parameters,
# in the future I might want to set more parameters
# note that centralities are calculated for the LCC by default which
# is the most reasonable choice; if you want to change that, set lcc to FALSE
require(NetCoMi)
require(tidyverse)
# calculate network stats -------------------------------------------------
# calculates net stats if possible, otherwise returns a try-error object.
# I am just using defaults for most parameters,
# in the future I might want to set more parameters
# note that centralities are calculated for the LCC by default which
# and the graphlet correlation matrix is returned by default
# is the most reasonable choice; if you want to change that, set lcc to FALSE
# to suppress the GCM glet = FALSE (useful in automated workflows) and ghm = FALSE
calculate_net_stats <- function(microNet_obj,
                                verbosePar = 1, lcc = TRUE,
                                glet  = TRUE, ghm = TRUE){
  netstats <- try({
    if(!inherits(microNet_obj, "microNet")){
      stop("microNet_obj is not a microNet object")
    }
    NetCoMi::netAnalyze(
      microNet_obj,
      centrLCC = lcc,
      avDissIgnoreInf = FALSE,
      sPathAlgo = "dijkstra",
      sPathNorm = TRUE,
      normNatConnect = TRUE,
      connectivity = TRUE,
      clustMethod = NULL,
      clustPar = NULL,
      clustPar2 = NULL,
      weightClustCoef = TRUE,
      hubPar = "eigenvector",
      hubQuant = 0.95,
      lnormFit = FALSE,
      weightDeg = FALSE,
      normDeg = TRUE,
      normBetw = TRUE,
      normClose = TRUE,
      normEigen = TRUE,
      graphlet = glet,
      gcmHeat = ghm,
      verbose = verbosePar
    )
  })
  return(netstats)
}

net_stats <- imap(MAN_inf_results, \(x, .y) calculate_net_stats(x, lcc = TRUE))
# TRUE if you want the LCC, which might be important for some network statistics



Note how at least some correaltion ar significant which is indicative of a network which does not have a random structure. You can actually extract and plot individual GCM like this (note that I am passing the parameters as a list, which is sometimes handy, and therefore I need to use `do.call()`:

In [ ]:
# plot a GCM from a microNetProps object
do.call(NetCoMi::plotHeat, list(
      mat  = net_stats[[1]]$graphletLCC$gcm1,
      pmat = net_stats[[1]]$graphletLCC$pAdjust1,
      type = "mixed",
      textCex = 0.8,
      labCex = 0.8,
      digits = 2,
      title = names(net_stats)[1]
    ))



## Tidyng up the results.  

I will now show a number of wrapper functions for tidying up the results of `netAnalyze()` and extracting them as data frames, which are easier to manipulate and use for plotting. Since by now you must have gotten used to how this can be extended to lists using functionals or loops, I will just work on the `net_stats` list and its elements.

### Extracting network properties.  

First, let's extract the network properties

In [ ]:
# extract_global_net_stats_df
# Extracts global network properties from a nested list of microNetProps
# objects (the output of calculate_net_stats / NetCoMi::netAnalyze).
#
# Arguments:
#   net_stats_list  Named list (one element per method). Each element is
#                   a microNetProps objects or try-error objects; non-microNetProps
#                   entries are silently skipped.
#
# Value:
#   A df containingtwo set of records (in teh variable called component):
#     all  globalProps  (whole network) + nnodes/ntaxa/nposedge/nnegedge
#     lcc  globalPropsLCC (largest connected component) + same extras
#
# requires: dplyr, purrr


extract_global_net_stats_df <- function(net_stats_one) {
  which_mnp <- purrr::map_lgl(net_stats_one, \(x) inherits(x, "microNetProps"))

  if (!any(which_mnp)) {
    return(list(global_all = tibble::tibble(), global_lcc = tibble::tibble()))
  }

  valid    <- net_stats_one[which_mnp]
  all_list <- vector("list", length(valid))
  lcc_list <- vector("list", length(valid))

  for (i in seq_along(valid)) {
    asso <- valid[[i]]$input$assoMat1
    lt   <- lower.tri(asso)
    extra <- c(
      nnodes   = sum(rowSums(asso != 0) > 1),
      ntaxa    = nrow(asso),
      nposedge = sum(asso[lt] > 0),
      nnegedge = sum(asso[lt] < 0)
    )
    all_list[[i]] <- c(unlist(valid[[i]]$globalProps),    extra)
    lcc_list[[i]] <- c(unlist(valid[[i]]$globalPropsLCC), extra)
  }
  names(all_list) <- names(lcc_list) <- names(valid)

  return_df <- list(
    all = dplyr::bind_rows(all_list, .id = "method") |>
      dplyr::mutate(dplyr::across(dplyr::where(is.numeric), \(x) dplyr::na_if(x, Inf))),
    lcc = dplyr::bind_rows(lcc_list, .id = "method") |>
      dplyr::mutate(dplyr::across(dplyr::where(is.numeric), \(x) dplyr::na_if(x, Inf)))
  ) |>
    list_rbind(names_to = "component")
}

net_stats_tidy     <- extract_global_net_stats_df(net_stats)
net_stats_tidy



This way you can easily join more information on the study itself and compare the properties of the networks obtained with different methods.  
Note the following:

*   SPIEC-EASI network is very fragmented and has low clustering
*   density is much higher with the two very lenient correlation based methods  
*   average path length is small, but only because these networks are small



### Extracting node properties.  

We could actually recalculate node properties for the tidygraphs, but let's try to extract them from the microNetProps objects; again, using a function makes things a bit tidier.


In [ ]:
# extract node properties
require(tidyverse)
require(phyloseq)


# extract node properties -------------------------------------------------
# a function to extract, as a data frame, the node properties from a
# "microNetProps" object (i.e. the output of calculate_net_stats / netAnalyze)
# nodestat is a data frame with per-taxon metadata (e.g. prevab_4),
# joined to the node properties on the "label" column

extract_node_stats <- function(net_stat_list){
  # assert diagonal is non-negative so the -1 correction for pos_degree is valid
  diag_vals <- diag(net_stat_list$input$assoMat1)
  if (any(diag_vals < 0)) {
    warning("extract_node_stats: negative values found on the diagonal of assoMat1 — ",
            "pos_degree calculation may be incorrect. Check your association matrix.")
  }
  if (any(diag_vals == 0)) {
    warning("extract_node_stats: zero values found on the diagonal of assoMat1 — ",
            "the -1 correction for pos_degree assumes a positive diagonal.")
  }
  node_props <- data.frame(
    pos_degree = apply(net_stat_list$input$assoMat1, 2, function(x) sum(x>0)-1),
    neg_degree = apply(net_stat_list$input$assoMat1, 2, function(x) sum(x<0)),
    clust_memb = net_stat_list[["clustering"]][["clust1"]],
    degree = net_stat_list[["centralities"]][["degree1"]],
    between = net_stat_list[["centralities"]][["between1"]],
    close = net_stat_list[["centralities"]][["close1"]],
    eigenv = net_stat_list[["centralities"]][["eigenv1"]]
  )
  node_props <- node_props |>
    dplyr::mutate(
      is_hub            = rownames(node_props) %in% net_stat_list$hubs$hubs1,
      pos_edge_fraction = pos_degree / (pos_degree + neg_degree),
      neg_edge_fraction = neg_degree / (pos_degree + neg_degree)
    ) |>
    tibble::rownames_to_column("label")
  return(node_props)
}

node_stats_df <- imap(net_stats, function(method, name) extract_node_stats(method)) |>
  list_rbind(names_to = "method")
head(node_stats_df)




In [ ]:
# let's join the taxonomy
taxtab_3 <- tax_table(physeq_to_use) |>
  as.data.frame()
colnames(taxtab_3) <- c("Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species")
taxtab_3 <- taxtab_3 |>
  as.data.frame() |>
  rownames_to_column("label")

node_stats_df <- left_join(node_stats_df, taxtab_3)
head(node_stats_df)

Now let's calculate relative median abundance for each genus and perform a bit of wrangling. We are going to use this later.

In [ ]:
# Calculate median relative abundance for each taxon and join
require(phyloseq)
require(tidyverse)
require(RColorBrewer)

# 1. Transform counts to relative abundance
amgut_genus_rel_abund <- phyloseq::transform_sample_counts(physeq_to_use, function(OTU) OTU / sum(OTU))

# 2. Extract the relative abundance table
OTU_counts_rel <- as(phyloseq::otu_table(physeq_to_use), "matrix")
# 3. calculate median rel. ab
median_OTU_ab <- apply(OTU_counts_rel, 1, \(x) median(x, na.rm = T)) |>
  as.data.frame() |>
  rownames_to_column("label")
colnames(median_OTU_ab)[2] <- "median_rel_ab"
# 4. join
node_stats_df <- dplyr::left_join(node_stats_df, median_OTU_ab)

# 5. get n top orders
n_orders_to_keep <- 11
orders_to_keep <- node_stats_df |> dplyr::select(label, Order, median_rel_ab) |>
  dplyr::distinct() |>
  dplyr::summarise(order_rel_ab = sum(median_rel_ab), .by = Order) |>
  arrange(desc(order_rel_ab))
top_n_orders <- dplyr::slice(orders_to_keep, 1:min(n_orders_to_keep, nrow(orders_to_keep))) |>
  pull(Order)
# 6. create a persistent palette for Orders, useful for comparing graphs
top_n_orders_palette <- brewer.pal(name = "Paired", n = length(top_n_orders)+1)
names(top_n_orders_palette) <- c(top_n_orders, "Other")

# 7. a bit of wrangling
node_stats_df_2 <- node_stats_df |>
  dplyr::mutate(top_order = if_else(Order %in% top_n_orders, Order, "Other")) |>
  dplyr::mutate(top_order = factor(top_order, levels = names(top_n_orders_palette)))
head(node_stats_df_2)


### A node plot.  

Especially in a large graph looking for "interesting" nodes might be complicated. Look at the node plot below, which is being used for the `spieceasi_mb` network only.  
You can easily adapt the code to create multiple plots and put them in a list.

In [ ]:
options(repr.plot.width = 11, repr.plot.height = 7)

# drawing a node plot
require(tidyverse)
require(ggrepel)
node_plot_method <- "spieceasi_mb"
temp_df <- node_stats_df_2 |>
    dplyr::filter(method == node_plot_method) |>
    dplyr::select(method, label, pos_degree, neg_degree, between,
                  Genus, median_rel_ab, top_order, is_hub) |>
    mutate(dgr = pos_degree + neg_degree) |>
    dplyr::filter(dgr > 0) |>
    mutate(
      tlabel    = str_trunc(Genus, 16, side = "center", ellipsis = "_")
    )
  if (nrow(temp_df) == 0) {
    warning(str_c("No data found for method '", node_plot_method, "' with degree > 0. Skipping plot."))
  }
  if ("Other" %in% levels(temp_df$top_order)) {
    temp_df <- mutate(temp_df,
                      top_order = fct_relevel(top_order, "Other", after = Inf))
  }

  ave_degree     <- mean(temp_df$dgr,        na.rm = TRUE)
  ave_pos_degree <- mean(temp_df$pos_degree, na.rm = TRUE)
  gtitle         <- str_c("Node stats, method ", node_plot_method, ".")

  n_orders       <- length(levels(temp_df$top_order))
  n_size_breaks  <- 4
  n_alpha_breaks <- 4

  p_plot <- ggplot(
    temp_df,
    mapping = aes(x = pos_degree, y = dgr)
  ) +
    geom_smooth(
      method      = "lm",
      linetype    = 3,
      se          = FALSE,
      color       = "black",
      show.legend = FALSE,
      fullrange   = FALSE
    ) +
    geom_point(
      mapping = aes(
        color = top_order,
        size  = median_rel_ab,
        shape = is_hub,
        alpha = between
      )
    ) +
    geom_text_repel(
      aes(label = tlabel),
      show.legend  = FALSE,
      max.overlaps = 20,
      alpha        = 0.5
    ) +
    geom_abline(slope = 1, intercept = 0, linetype = 1) +
    geom_hline(yintercept = ave_degree,     linetype = 3) +
    geom_vline(xintercept = ave_pos_degree, linetype = 3) +
    scale_x_continuous(
      limits = c(0, NA),
      expand = expansion(mult = c(0, 0.05))
    ) +
    scale_y_continuous(
      limits = c(0, NA),
      expand = expansion(mult = c(0, 0.05))
    ) +
    scale_alpha_continuous(range = c(0.4, 1)) +
    scale_color_manual(values = top_n_orders_palette) +
    scale_size_continuous(range = c(1, 6)) +

    labs(
      x     = "positive degree",
      y     = "degree",
      size  = "median rel. abundance",
      alpha = "betweenness",
      color = "Order",
      shape = "is hub?",
      title = gtitle
    ) +
    guides(
      color = guide_legend(
        ncol  = if (n_orders  > 4) 2 else 1,
        byrow = TRUE,
        order = 1
      ),
      size = guide_legend(
        ncol  = if (n_size_breaks  > 3) 2 else 1,
        byrow = TRUE,
        order = 3
      ),
      alpha = guide_legend(
        ncol  = if (n_alpha_breaks > 3) 2 else 1,
        byrow = TRUE,
        order = 4
      )
    ) +
    theme_bw() +
    theme(
      plot.title      = element_text(hjust = 0.5),
      legend.box      = "vertical",
      legend.key.size = unit(0.6, "cm"),
      legend.text     = element_text(size = 8),
      legend.title    = element_text(size = 9)
    )

p_plot


Right click on the graph to open a contextual menu and open it in another browser window to explore it better.  
The node plots provide a number of visual cues:

- the **horizontal and vertical dotted lines** show the average value for degree and positive degree and may help in identifying nodes whose values are far from the mean;

- the **diagonal continuous line** corresponds to points for which positive degree and degree are equal; points above the line have at least 1 negative interaction

- the **diagonal dotted line** is the linear regression line for degree as a function of positive degree; its position and slope relative to the previous line indicates on average how much the ratio between total degree and positive degree changes as a function of degree; data points which are far from this line also have an unusual ratio between the two values compared to the other nodes in the same dataset.

- **negative hubs** will be located in the upper half above the dotted lines; **positive hubs** will be located in the right.  

- **the shape of the symbols** idicates if `netAnalyze()` classified the node as a hub  

- **size** is proportional to median relative abundance, but this might be ambiguous in sparse OTU tables

There is a lot of space for improvement (but we do not have the time to go though this):  

- **can we find a more statistically principled way to identify hubs**? `netAnalyze()` itself offers two different options, one of which uses quantiles of the lognormal distribution of node degree (which is reasonable in most cases)  

- **can we identify a better way to mark nodes with anomalous PEP values**: one might interpret nodes with high negative degree as competitors, because they tend to exclude other nodes, may be dominating one or more niches in the ecosystem, and the others might be generalists (but this needs to be proven...)

- one can do only so much with perceptually interesting features in a graph: can we find a better way to highligh betweeness: a node with high betweenness is possibly a bottleneck, and its removal might fragment the network

# Plotting the network.  

Network visualization is central to presentation of results of a Microbial Association network workflow. NetCoMi itself has a very powerful (plot())[https://netcomi.de/reference/plot.micronetprops] command, which takes as an input a microNetProps object. As usual it has tens of parameters and, most important of all, it produces a base R plot.
Usually, these are the issues you should thnk about when plotting a network:

*   **which layout should I use**: layouts determine the way nodes and edges are distributed on a cartesian plane or, sometimes, even in three dimensions. Force-based layouts, which brings close nodes which are strongly connected(based on the weights of the edges) against a general repulsive force are frequently used with Fruchterman-Reingonld being very prominent in MAN representation; Kamada-Kawai is a useful alternative
*   **which features should I map to nodes?**
    + **labels**: taxa or ASV/OTU names are an option, but they can make the graph very crowded. Think at ways you can show only the most prominent ones. Also think about features of the labels: font, size, color, style...
    + **color**: color can be mapped to several variables but using cluster membership or some higher taxonomic level is common; quantitative features can be also mapped to continuous color scales
    + **transparencey** or alpha: mappet to some other quantitative feature
    + **size**: usually mapped to some centrality value (eigenvector, closeness and degree are common)
    + **shape**: usually matched to some other feature; for example, in a cross-domain network, could be the domain
*   **which features should I map to edges**:
    + **color**: usually green for copresence edges and red for mutual exclusion, but this is obviously not color-blind safe
    + **line style**: could replace the color but might be difficult to perceive in crowded networks
    + **thickness**: usually associated to edge weight (which is usuallyunsigned strentgh of the association), but might be mapped to other edge centralities.

An excellent article on network visualization is [here](https://kateto.net/network-visualization/)

## Plotting with NetCoMi.

I will simply use the example from NetCoMi [get started page](https://netcomi.de/articles/netcomi) on the `spieceasi_mb` MAN.

In [ ]:
require(NetCoMi)
options(repr.plot.width = 7, repr.plot.height = 7)
p <- plot(MAN_inf_simple_wstats,
          labelScale = FALSE,
          nodeColor = "cluster",
          nodeSize = "eigenvector",
          rmSingles = TRUE,
          title1 = "Network on genus level with SPIEC-EASI associations",
          showTitle = TRUE,
          cexTitle = 1.0,
          cexLabels = 0.8)

legend(x = 0.35, y = 0.9, cex = 1.0, title = "estimated association:",
       legend = c("+","-"), lty = 1, lwd = 3, col = c("#009900","red"),
       bty = "n", horiz = TRUE)


You need to believ me if I tell you that cohercing this to fit in a graphic device does require some trial and error.

## Plotting with tidygraph/ggraph.

Even if the documentation of [tidygraph](https://tidygraph.data-imaginist.com/) and [ggraph](https://ggraph.data-imaginist.com/) is a bit cryptic both are excellent tools for graph manipulation and visualization. Even if `tidygraph` lacks some of the tooks available in `igraph` the objects it produces inherit the `igraph` class so they can be fed to `igraph` functions directly.

In [ ]:
# this is just so that you know
require(NetCoMi)
require(tidyverse)
require(tidygraph)
# convert microNetProps objects to tidygraphs
tg <- microNet_to_tidygraph(MAN_inf_simple)
# tidygraph has a very simple syntax for accessing and transforming the nodes and edges
str(tg)
tg_nodes <- tg |> activate(nodes) |> as_tibble()
tg_edges <- tg |> activate(edges) |> as_tibble()
head(tg_nodes)
head(tg_edges)

You may have noted that the nodes do not have any stats attached to them. We could easily calculate them in tidygraph. Look at these examples for [calculating centralities](https://tidygraph.data-imaginist.com/reference/centrality.html) or [clustering](https://tidygraph.data-imaginist.com/reference/group_graph.html). In fact, we will levarage on this when recalculating stats for co-presence networks.
Right now it is much easier to use what we already have in the `node_stats_df`, whch we need to filter for `spieceasi_mb`.

In [ ]:
require(tidyverse)
require(tidygraph)

# merge node stats

node_stats_to_tg <- node_stats_df_2 |>
  dplyr::filter(method == "spieceasi_mb") |>
  dplyr::select(-method)
  # The 'label' column was prematurely renamed to 'name' here,
  # but the 'merge_stats' function expects 'label' to be present.
  # The line 'rename(name = label)' is removed from here.
head(node_stats_to_tg)
tg_w_stats <- tg |>
  activate(nodes) |>
  left_join(node_stats_to_tg, by = c("name" = "label"))
tg_w_stats |> activate(nodes) |> as_tibble() |> head(10)

# or with a function
merge_stats <- function(tg, node_stats = FALSE, ebetw = TRUE){
  # check required packages are installed
  required_pkgs <- c("tidygraph", "dplyr")
  missing_pkgs <- required_pkgs[!vapply(required_pkgs, requireNamespace, logical(1), quietly = TRUE)]
  if (length(missing_pkgs) > 0) {
    stop(
      "The following packages are required but not installed: ",
      paste(missing_pkgs, collapse = ", "),
      "\nInstall with: install.packages(c(",
      paste0('"', missing_pkgs, '"', collapse = ", "), "))"
    )
  }
  if (!tidygraph::is.tbl_graph(tg)) stop("This is not a tidygraph")
  if (ebetw) {
    tg <- tg |>
      tidygraph::activate(edges) |>
      dplyr::mutate(edge_betw = tidygraph::centrality_edge_betweenness())
  }
  # check if node_stats is a data frame to decide whether to merge nodes
  if (is.data.frame(node_stats)) {
    # rename label to name to match the tidygraph node column before joining
    node_stats <- dplyr::rename(node_stats, name = label)
    tg <- tg |>
      tidygraph::activate(nodes) |>
      dplyr::left_join(node_stats, by = "name") |>
      dplyr::mutate(tot_degree = pos_degree + neg_degree)
  }
  class(tg) <- c("tg_wstats", class(tg))
  return(tg)
}

tg_w_stats <- merge_stats(tg, node_stats_to_tg)
tg_w_stats |> activate(nodes) |> as_tibble() |> head(10)

We are now ready for plotting with ggraph. Again, I will use a few options to standardize the plot but you should check [`ggraph` reference](https://ggraph.data-imaginist.com/reference/index.html) for more options.

In [ ]:
# setting plot size
options(repr.plot.width = 12, repr.plot.height = 12)
# this is just so that you know, but the function does its own checks
require(tidyverse)
require(tidygraph)
require(ggraph)
require(randomcoloR)
# variable to use for coloring the nodes
color_by <- "top_order"
# variable to use for naming the nodes
node_label <- "Genus"
# layout to use
# layout: network layout algorithm; one of:
#   - "fr": Fruchterman-Reingold (default) — force-directed, most common in microbiome network papers
#   - "kk": Kamada-Kawai — force-directed, often cleaner for smaller or denser networks
#   - "stress": stress minimization (graphlayouts) — deterministic and reproducible
layout <- "fr"
# alpha for nodes
a <- 0.6
# clp: coord_cartesian clip (default "off", alternative "on")
clp <- "off"
# legend position
lp <- "right"
# name max length, otherwise will be clipped
name_length <- 15
# a few color blid safe schemes for edges
# Define Palettes for edges
palette_eco_safe     <- c(copres = "#009E73", mut_ex = "#D55E00")
palette_teal_coral   <- c(copres = "#008080", mut_ex = "#FF6F59")
palette_cyan_magenta <- c(copres = "#00A598", mut_ex = "#CC79A7") # Print-safe variants
palette_cyan_magenta_2 <- c(copres = "cyan", mut_ex = "magenta")
palette_green_red <- c(copres = "green", mut_ex = "red") # the standard palette for MAN
edge_cols <- palette_teal_coral
# remove nodes with 0 degree; make cluster membership a factor
g2plot <- tg_w_stats |>
    tidygraph::activate(nodes) |>
    dplyr::filter(pos_degree > 0 | neg_degree > 0) |>
    dplyr::mutate(clust_memb = forcats::as_factor(clust_memb))

g2plot_title <- paste("amgut_filt_renamed", "spieceasi_mb", sep = " ")
ggraph_plot <- ggraph::ggraph(g2plot, layout = layout) +
      ggraph::geom_edge_link(
        mapping = ggplot2::aes(edge_colour = asso_type, edge_width = weight),
        alpha = 0.5,
        show.legend = TRUE
      ) +
      ggraph::geom_node_point(mapping = ggplot2::aes(colour = .data[[color_by]],
        size = tot_degree)) +
      ggraph::geom_node_text(
        mapping = ggplot2::aes(
          label = stringr::str_trunc(
            string = .data[[node_label]],
            width = name_length,
            side = "center",
            ellipsis = "."
          )
        ),
        check_overlap = FALSE,
        alpha = a
      ) +
      labs(title = g2plot_title, size = "degree") +
      ggraph::scale_edge_color_manual(values = edge_cols) +
      ggraph::scale_edge_width_continuous(range = c(1, 4)) +
      scale_size_continuous(range = c(2, 10)) +
      coord_cartesian(clip = clp) +
      ggraph::theme_graph() +
      theme(plot.title = element_text(hjust = 0.5),
            legend.position = lp)

# explicit discrete color scale sized to the number of levels actually present,
# avoiding RColorBrewer's hard palette-size limits (e.g. Accent maxes at 8).
# This applies to both phylum and clust_memb, since cluster counts in particular
# can easily exceed 8 communities.
n_levels <- g2plot |>
      tidygraph::activate(nodes) |>
      tibble::as_tibble() |>
      dplyr::pull(.data[[color_by]]) |>
      droplevels() |>
      nlevels()

if (n_levels <= 8) {
    ggraph_plot <- ggraph_plot +
      scale_color_brewer(palette = "Accent")
} else {
    set.seed(6317)
    ggraph_plot <- ggraph_plot +
      scale_color_manual(values = randomcoloR::randomColor(n_levels, luminosity = "dark"))
}


 ggraph_plot

Explore with different options (layouts, showing or hiding legends, etc.)

### Extracting the copresence network.  

Using tidygraph makes it easier to extract the co-presence network and to recalculate statistics: this is necessary because all statistics in microNetPropsobjects are calculated on all association types and, to detect co-presence clusters and keystones, they need to be recalculated.



In [ ]:

tg_w_stats |> activate(nodes) |> as_tibble() |> head(10)

In [ ]:
require(tidygraph)

extract_copres_net <- function(tg, q_threshold = 0.90){
  # 1. check required packages
  required_pkgs <- c("tidygraph", "dplyr")
  missing_pkgs <- required_pkgs[!vapply(required_pkgs, requireNamespace, logical(1), quietly = TRUE)]
  if (length(missing_pkgs) > 0) {
    stop(
      "The following packages are required but not installed: ",
      paste(missing_pkgs, collapse = ", "),
      "\nInstall with: install.packages(c(",
      paste0('"', missing_pkgs, '"', collapse = ", "), "))"
    )
  }
  # 2. keep copresence edges only
  tg <- tg |> activate(edges) |>
    dplyr::filter(asso_type == "copres") |>
    dplyr::select(-edge_betw)
  # 2. recalculate node stats, using weights, use pos_degree and neg_degree for
  # coherence with plot_ggraph
  tg <- tg |>
    activate(nodes) |>
    dplyr::select(name:nlabel, Kingdom:top_order) |>
    mutate(degree = centrality_degree(weights = weight, normalized = T)) |>
    dplyr::filter(degree>0) |>
    mutate(degree = centrality_degree(weights = abs(asso_est), normalized = T)) |>
    dplyr::filter(degree > 0) |>
    mutate(pos_degree = degree,
           neg_degree = 0,
           tot_degree = degree,
           between = centrality_betweenness(weights = abs(asso_est), directed = F, normalized = T),
           close = centrality_closeness(weights = abs(asso_est), normalized = T),
           eigen = centrality_eigen(weights = abs(asso_est), scale = T),
           clust_memb = as.factor(group_louvain(weights = abs(asso_est)))
    ) |>
    mutate(is_high_degree = degree >= quantile(degree, q_threshold, na.rm = TRUE),
           is_high_close = close >= quantile(close, q_threshold, na.rm = TRUE),
           is_high_eigen = eigen >= quantile(eigen, q_threshold, na.rm = TRUE),
           is_hub = (is_high_degree + is_high_close + is_high_eigen) >= 2)
  # 3. recalculate edge betweness
  tg <- tg |>
    activate(edges) |>
    mutate(edge_betw = centrality_edge_betweenness(weights = weight, directed = F))
  return(tg)
}

copres_network <- extract_copres_net(tg_w_stats)
copres_network |> activate(nodes) |> as_tibble() |> head(10)
copres_network |> activate(edges) |> as_tibble() |> head(10)



In [ ]:
# setting plot size
options(repr.plot.width = 12, repr.plot.height = 12)
# this is just so that you know, but the function does its own checks
require(tidyverse)
require(tidygraph)
require(ggraph)
require(randomcoloR)
# color by cluster
color_by_2 <- "clust_memb"

# remove nodes with 0 degree; make cluster membership a factor
copres2plot <- copres_network |>
    tidygraph::activate(nodes) |>
    dplyr::mutate(clust_memb = forcats::as_factor(clust_memb))

copresplot_title <- paste("amgut_filt_renamed, copresence", "spieceasi_mb", sep = " ")
cop_ggraph_plot <- ggraph::ggraph(copres2plot, layout = layout) +
      ggraph::geom_edge_link(
        mapping = ggplot2::aes(edge_colour = asso_type, edge_width = weight),
        alpha = 0.5,
        show.legend = TRUE
      ) +
      ggraph::geom_node_point(mapping = ggplot2::aes(colour = .data[[color_by_2]],
        size = tot_degree)) +
      ggraph::geom_node_text(
        mapping = ggplot2::aes(
          label = stringr::str_trunc(
            string = .data[[node_label]],
            width = name_length,
            side = "center",
            ellipsis = "."
          )
        ),
        check_overlap = FALSE,
        alpha = a
      ) +
      labs(title = g2plot_title, size = "degree") +
      ggraph::scale_edge_color_manual(values = edge_cols) +
      ggraph::scale_edge_width_continuous(range = c(1, 4)) +
      scale_size_continuous(range = c(2, 10)) +
      coord_cartesian(clip = clp) +
      ggraph::theme_graph() +
      theme(plot.title = element_text(hjust = 0.5),
            legend.position = lp)

# explicit discrete color scale sized to the number of levels actually present,
# avoiding RColorBrewer's hard palette-size limits (e.g. Accent maxes at 8).
# This applies to both phylum and clust_memb, since cluster counts in particular
# can easily exceed 8 communities.
n_levels <- g2plot |>
      tidygraph::activate(nodes) |>
      tibble::as_tibble() |>
      dplyr::pull(.data[[color_by]]) |>
      droplevels() |>
      nlevels()

if (n_levels <= 8) {
    ggraph_plot <- ggraph_plot +
      scale_color_brewer(palette = "Accent")
} else {
    set.seed(6317)
    ggraph_plot <- ggraph_plot +
      scale_color_manual(values = randomcoloR::randomColor(n_levels, luminosity = "dark"))
}


 cop_ggraph_plot

Two very distinct enterotypes are visible here.  

## Is it worth the effort?  

As you may have noticed R is not exactly an interactive network visualization tool. If you like to play around using menus and pointing and clicking (and being irreproducible in the process), consider exporting the graph to a format which can be used in [Gephi](https://gephi.org/) or [Cytoscape](https://cytoscape.org/), both are excellent Graph Viz platforms.
You have several options. Among these:  


*   export the edge and node tables as .csv or .txt files and reimport them in the Graph Viz software
*   export the network as a GraphML file: it usually works with both Gephi and Cytoscape



In [ ]:
require(igraph)
igraph::write_graph(igraph_se, file = "myigraphnetwork.graphml", format = "graphml")
colab_msg(text = "Network saved as .gml", type = "success")
colab_msg(text = "Don't forget to download the file or save it to your Google Drive before you close the session", type = "warning")

# More.  

We really do not have the time to do more in this lecture. Things that my deserve more attention are:  

1.   **comparing networks**: this is one of the main reasons why learning how to use NetCoMi is worth the effort. You can learn more in the [NetCoMi documentation](https://netcomi.de/articles/net_comparison). Network comparison can help you generating and testing biologically meaningful hypotheses. You may also want to check out the NetCoMi article and this paper: Liu, C., Li, C., Jiang, Y., Zeng, R.J., Yao, M., Li, X., 2023. A guide for comparing microbial co-occurrence networks. iMeta 2. https://doi.org/10.1002/imt2.71.
2.   **inferring cross-domain (or cross-kingdom) networks**: in food, host and environmental microbiomes interaction between entities belonging to different Domains or Kingdoms are important (Plant-Fungi and Plant-Bacteria in the rhizosphere; Fungi-Bacteria in many situations). There is a specific function in the [SpiecEasi package](https://github.com/zdk123/SpiecEasi#cross-domain-interactions) and the inference of cross-domain networks with NetCoMi is [documented here](https://netcomi.de/articles/cross_domain_networks). Here are a few important papers:  
    * Tipton, L., Müller, C.L., Kurtz, Z.D., Huang, L., Kleerup, E., Morris, A., Bonneau, R., Ghedin, E., 2018. Fungi stabilize connectivity in the lung and skin microbial ecosystems. Microbiome 6, 12. https://doi.org/10.1186/s40168-017-0393-0  
    * Lee, K.K., Kim, H., Lee, Y.-H., 2022. Cross-kingdom co-occurrence networks in the plant microbiome: Importance and ecological interpretations. Front. Microbiol. 13, 953300. https://doi.org/10.3389/fmicb.2022.953300  
    * Brunner, J.D., Robinson, A.J., Chain, P.S.G., 2024. Combining compositional data sets introduces error in covariance network reconstruction. ISME Commun. 4, ycae057. https://doi.org/10.1093/ismeco/ycae057  
3. the validity of MAN inference itself has been discussed several times: I leave this to you and your favourite LLM. I can only advise care and a sizable amount of critical thinking when dealing with this.

*Finis. Pro bono malum*




# Sandbox.  

This is a mini sandbox for exploring objects in the environmenta and saving them.

In [ ]:
# mini sandbox

# 1. Define the target class you want to filter objects in the environment for
target_class <- "phyloseq"

# 2. Filter the environment
all_obj_names <- ls()
matching_indices <- sapply(mget(all_obj_names), function(x) inherits(x, target_class))
filtered_objects <- mget(all_obj_names[matching_indices])

# 3. View the results
print(names(filtered_objects))

# 4. save filtered objects
save_data <- FALSE
if(save_data){
  save(filtered_objects, file = "my_filtered objects.RData")
  colab_msg(text = "Filtered objects saved", type = "success")
  colab_msg(text = "Don'tforget to download the fileor save it to your Google Drive before you close the session", type = "warning")
}

# 5. save the whole environment
save_environment <- FALSE
if(save_environment){
  save.image(file = "complete_r_environment.RData")
  colab_msg(text = "Session environment saved", type = "success")
  colab_msg(text = "Don'tforget to download the fileor save it to your Google Drive before you close the session", type = "warning")
  }

# References.  

*   Ghaeli, Z., Aghdam, R., Eslahchi, C., 2025. Evaluating microbial network inference methods: Moving beyond synthetic data with reproducibility-driven benchmarks. bioRxiv 8, 2025.07.05.663212. https://doi.org/10.1101/2025.07.05.663212
*   Kurtz, Z.D., Müller, C.L., Miraldi, E.R., Littman, D.R., Blaser, M.J., Bonneau, R.A., 2015. Sparse and compositionally robust inference of microbial ecological networks. PLoS computational biology 11, e1004226. <https://doi.org/10.1371/journal.pcbi.1004226>.
*   Liu, C., Li, C., Jiang, Y., Zeng, R.J., Yao, M., Li, X., 2023. A guide for comparing microbial co-occurrence networks. iMeta 2. <https://doi.org/10.1002/imt2.71>
*   Parente, E., Zotta, T., Ricciardi, A., 2022. A review of methods for the inference and experimental confirmation of microbial association networks in cheese. Int. J. Food Microbiol. 368, 109618. https://doi.org/10.1016/j.ijfoodmicro.2022.109618
*   Peschel, S., Müller, C.L., Mutius, E. von, Boulesteix, A.-L., Depner, M., 2020. NetCoMi: network construction and comparison for microbiome data in R. Brief Bioinform. <https://doi.org/10.1093/bib/bbaa290>
*   Tackmann, J., Rodrigues, J.F.M., Mering, C. von, 2019. Rapid Inference of Direct Interactions in Large-Scale Ecological Networks from Heterogeneous Microbial Sequencing Data. Cell Syst 9, 286-296.e8. https://doi.org/10.1016/j.cels.2019.08.002
*   Yaveroğlu, Ö.N., Malod-Dognin, N., Davis, D., Levnajic, Z., Janjic, V., Karapandza, R., Stojmirovic, A., Pržulj, N., 2014. Revealing the Hidden Language of Complex Networks. Sci. Rep. 4, 4547. https://doi.org/10.1038/srep04547
*   Wang, M., Wang, H., Zheng, H., 2022. A Mini Review of Node Centrality Metrics in Biological Networks. Int. J. Netw. Dyn. Intell. 99–110. https://doi.org/10.53941/ijndi0101009




# Miscellaneous.  

## Does it work?   

I have tested this using

*   Google Chrome Version 148.0.7778.180 (Official build) (arm64)

*   Brave browser Release Notes v1.90.124 (May 20, 2026) (arm64)

*   Safari 26.5 (21624.2.5.11.4)

and, as far I can see it works



## Version.  
This is version 2.1, 29/06/2026


# Credits and copyright.

## Credits.  
Most of the code for the creation and restoring of libraries from Google Drive and for installing GitHub packages in a CoLab notebook was produced by Google Gemini AI with my prompts.  Some inconsistencies were then fixed with Claude Sonnet 4.6

MIT License


## Copyright.  
Copyright (c) [2026] [Eugenio Parente]

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

# Miscellaneous.  

## Does it work?   

I have tested this using

*   Google Chrome Version 148.0.7778.180 (Official build) (arm64)

*   Brave browser Release Notes v1.90.124 (May 20, 2026) (arm64)

*   Safari 26.5 (21624.2.5.11.4)

and, as far I can see it works



## Version.  
This is version 2.1, 29/06/2026


# Credits and copyright.

## Credits.  
Most of the code for the creation and restoring of libraries from Google Drive and for installing GitHub packages in a CoLab notebook was produced by Google Gemini AI with my prompts.  Some inconsistencies were then fixed with Claude Sonnet 4.6

MIT License


## Copyright.  
Copyright (c) [2026] [Eugenio Parente]

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.